# 02 — Klasik Baseline (SUBMIT-HAZIR): detection + linking + gap-fill

Detection = **Otsu → local-max → `center_of_mass`**. Linking = **Hungarian 8 µm**.
Post-process = **gap-fill** (1-kare boşluk kapatma). Sıfır ek bağımlılık.

### Konfigürasyon (ölçülerek sabitlendi)
- **`GAPFILL=True`** — DEV'de resmi metrikle doğrulandı: `adj_edge_J 0.6918 → 0.7014` (+36 TP / +14 FP). ✅
- **`DIV_ON=False`** — resmi metrikte bölünme net zararlı (edge −0.0034, div_J=0); kod öneri 6 için duruyor.
- **Ceza-farkında yerel metrik** (`metrics.md` ile doğrulandı): resmi FP kuralı + video-başına ceza.
- **Güvenli kurulum** (`sys.path.append`) + sağlam `find_root`.

### Denendi, ELENDİ (hepsi doğru metrikle ölçüldü)
- ❌ watershed (dense recall 0.88→0.35) · ❌ thr/FOOT tuning (overfit, LB 0.749→0.739)
- ❌ motion-model linking (GT hız cos 0.30 → öngörülemez) · ❌ gate büyütme (GT hareket <8µm)
- ❌ confidence-filter (recall kaybı FP kazancını yiyor) · ❌ sınır temizleme · ❌ top-N

> ### ⚠️ SUBMIT KURALLARI
> - **Internet KAPALI** (Settings → Internet = Off). `pip install` YOK.
> - **`erdeemt/cell-tracking-libs`** + **yarışma verisi** — ikisi de EKLİ.
> - Code competition: gizli test runtime'da bağlanır → test isimleri **hardcode edilmez**.
> - Çıktı: **`/kaggle/working/submission.csv`**. (DEV analiz hücreleri scored rerun'da `DEV=False` ile atlanır.)

## 0 — Kurulum (zarr utility dataset'ten)

In [ ]:
# zarr bu imajda YOK -> erdeemt/cell-tracking-libs (pylibs) sys.path ile.
# append! insert(0) DEGIL: pylibs'te numpy 2.5.1 var, ortamin 2.0.2'sini golgelememeli
# (scipy/skimage 2.0.2'ye karsi derlenmis).
import sys, os, time, warnings
from pathlib import Path
from collections import Counter

LIBS = Path('/kaggle/input/datasets/erdeemt/cell-tracking-libs/pylibs')
if not LIBS.exists():
    hits = [Path(r) for r, d, f in os.walk('/kaggle/input') if os.path.basename(r) == 'pylibs']
    assert hits, 'cell-tracking-libs dataset i notebook a ekli degil!'
    LIBS = hits[0]
sys.path.append(str(LIBS))

import numpy as np
import pandas as pd
import zarr
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.filters import threshold_otsu
warnings.filterwarnings('ignore')

print('zarr ', zarr.__version__, '| numpy', np.__version__)
assert np.__version__.startswith('2.0'), f'numpy golgelendi: {np.__version__} — insert(0) mi kullandin?'
print('hazir')

## 1 — Yapılandırma & veri kökü

In [ ]:
SCALE_ZYX = (1.625, 0.40625, 0.40625)   # um/piksel (Z,Y,X) — zarr attrs'ten dogrulandi
S = np.array(SCALE_ZYX, dtype=np.float32)

LINK_MAX_UM = 8.0      # linking arama yaricapi (EDA: hareket 99p ~8 um)
MATCH_UM    = 7.0      # metrik eslesme toleransi

# Detection (GUVENLI/kanitlanmis degerler — LB 0.749). Esigi/footprint'i degistirmek
# 4 placeholder'a overfit ediyor; prefix-bazli CV olmadan DOKUNMA.
SIGMA = (1, 2, 2)      # Gauss yumusatma (voxel; Z ekseni ince)
FOOT  = (3, 11, 11)    # local-max footprint ~ cekirdek boyutu (~10 um)

# Bolunme (division) — KAPALI (resmi metrikte net zararli; oneri 6 icin kod kaliyor).
DIV_ON     = False
DIV_MAX_UM = 6.0
DIV_SIB_UM = 13.0

# Gap-fill (1-kare bosluk kapatma): track t'de biter, baska track t+2'de baslarsa ve yakinsa
# aradaki t+1'e sentetik node koy, iki kenari kurtar. Miss-analizinde tavan +76 kenar; ama
# GT'siz calisir -> bazi yanlis kopruler (FP). GERCEK kazanc DEV'de olculecek (off/on).
GAPFILL    = True
GAP_MAX_UM = 10.0      # 2-kare yer degistirme esigi (hareket ~2-4um/kare)

TTRUE_REF_FRAMES = 6   # yerel metrikte ceza icin T_true tahmini (ornek kare sayisi)
MAX_TEST = None        # None = tum test datasetleri (gercek rerun icin SART)
OUT_CSV  = Path('/kaggle/working/submission.csv')
INPUT    = Path('/kaggle/input')
COMP_DIR = INPUT / 'competitions' / 'biohub-cell-tracking-during-development'

def find_root():
    """train/ + test/ iceren yarisma dizinini bul (dogrulanmis yol + arama + teshis)."""
    if (COMP_DIR / 'train').is_dir() and (COMP_DIR / 'test').is_dir():
        return COMP_DIR
    st = [(INPUT, 0)]
    while st:
        b, d = st.pop()
        try:
            if (b / 'train').is_dir() and (b / 'test').is_dir():
                return b
        except Exception:
            pass
        if d < 5:
            try:
                for c in sorted(b.iterdir()):
                    if c.is_dir() and not c.name.endswith(('.zarr', '.geff')):
                        st.append((c, d + 1))
            except Exception:
                pass
    top = sorted(p.name for p in INPUT.iterdir()) if INPUT.exists() else []
    raise RuntimeError('yarisma koku bulunamadi (train/+test/ yok). Add Input ile yarisma '
                       'verisini EKLE. /kaggle/input: ' + str(top))

ROOT = find_root(); TRAIN = ROOT / 'train'; TEST = ROOT / 'test'
test_names = sorted(p.stem for p in TEST.glob('*.zarr'))
print('ROOT:', ROOT)
print(f'train={len(list(TRAIN.glob("*.zarr")))} | test={len(test_names)}')
print('test:', test_names[:10], '...' if len(test_names) > 10 else '')

# DEV mi GERCEK RERUN mu? Placeholder test = train'den kopya -> GT'leri train'de.
# Gercek rerun'da gizli test isimleri train'de YOK -> tum dogrulama/eval atlanir.
DEV = len(test_names) > 0 and (TRAIN / (test_names[0] + '.geff')).exists()
print('\nMOD:', 'DEV (placeholder test, GT var)' if DEV else 'GERCEK RERUN (gizli test, GT yok)')

## 2 — Okuyucular

In [ ]:
def open_image(zpath):
    """OME-Zarr goruntu dizisini ac -> (T,Z,Y,X)."""
    n = zarr.open(str(zpath), mode='r')
    a = dict(n.attrs)
    ms = a.get('multiscales') or (a['ome'].get('multiscales') if isinstance(a.get('ome'), dict) else None)
    if ms:
        return n[ms[0]['datasets'][0]['path']]
    return n['0'] if '0' in list(n.keys()) else n

def load_geff(gp):
    """GEFF -> (nodes_df[id,t,z,y,x], edges (E,2)). Duz zarr; geff/tracksdata gerekmiyor."""
    g = zarr.open(str(gp), mode='r')
    d = {'id': np.asarray(g['nodes/ids'])}
    for k in ('t', 'z', 'y', 'x'):
        d[k] = np.asarray(g[f'nodes/props/{k}/values'])
    return pd.DataFrame(d), np.asarray(g['edges/ids'])

print('ok')

## 3 — Detection: Otsu → local-max → ağırlık merkezi

Gauss yumuşatma → Otsu eşiği → çekirdek-boyutlu local-max → bağlı bileşen `center_of_mass`.
Her çekirdeğin parlak merkezi = bir tespit (dense dokuda bile bir-çekirdek-bir-marker).
Alt-voxel merkez, ham peak voxel'ine kıyasla çift-tespiti önler (linking'i korur).

In [ ]:
def detect_frame(v):
    sm = ndi.gaussian_filter(v.astype(np.float32), sigma=SIGMA)
    thr = threshold_otsu(sm)
    mx = ndi.maximum_filter(sm, size=FOOT)
    peaks = (sm == mx) & (sm > thr)
    lbl, n = ndi.label(peaks)
    if n == 0:
        return np.zeros((0, 3), np.float32)
    return np.asarray(ndi.center_of_mass(sm, lbl, np.arange(1, n + 1)), np.float32)  # (N,3) voxel


def detect_all(arr):
    return [detect_frame(np.asarray(arr[t])) for t in range(arr.shape[0])]


def estimate_true_count(arr, frames=TTRUE_REF_FRAMES):
    """Yerel metrikte ceza icin T_true proxy = referans blob sayimi × T.
    Referans = mevcut detector oldugundan bu config'te penalty ~1.0 (dogru: fazla-tahmin yok).
    Detection deneyi yaparken (CV) referansi bu FOOT11/Otsu hattina SABITLE ki compass olsun."""
    T = arr.shape[0]
    ts = np.linspace(0, T - 1, min(frames, T)).astype(int)
    cnts = [len(detect_frame(np.asarray(arr[int(t)]))) for t in ts]
    return int(np.median(cnts) * T)


# hiz + yogunluk kontrolu (T'yi HARDCODE ETME)
_arr = open_image(TEST / (test_names[0] + '.zarr'))
_T = _arr.shape[0]
t0 = time.time(); _c = detect_frame(np.asarray(_arr[_T // 2])); dt = time.time() - t0
print(f'shape {_arr.shape} | 1 kare {dt:.2f}s | cekirdek {len(_c)}')
print(f'tahmini 1 dataset ({_T} kare): {dt*_T:.0f}s | {len(test_names)} test: {dt*_T*len(test_names)/60:.1f} dk')

## 4 — Linking (Hungarian 8 µm) + bölünme + gap-fill

**Linking:** ardışık karelerde optimal 1-1 eşleştirme; 8 µm üstü yasak. Eşleşmeyen = beliriş/kayboluş.
**Bölünme** (`DIV_ON`, şu an KAPALI): eşleşmemiş node'u zaten-çocuklu ebeveyne bağlar.
**Gap-fill** (`GAPFILL=True`): `t`'de biten track + `t+2`'de başlayan track ≤10 µm ise, aradaki
`t+1`'e sentetik node koyup iki kenarı kurtarır — **GT gerekmez** (inference'ta çalışır). 1-kare
detection kaçırmalarını telafi eder; DEV'de +36 TP / +14 FP ölçüldü (net +0.0096).

In [ ]:
def link_pairs(A, B):
    if len(A) == 0 or len(B) == 0:
        return [], None
    D = cdist(A * S, B * S)                        # um mesafe
    cost = np.where(D <= LINK_MAX_UM, D, 1e6)      # kapi
    r, c = linear_sum_assignment(cost)
    return [(int(i), int(j)) for i, j in zip(r, c) if D[i, j] <= LINK_MAX_UM], D


def link_and_divide(A, B, div_on):
    pairs, D = link_pairs(A, B)
    if not div_on or D is None or not pairs:
        return pairs, []
    child_of = {i: j for i, j in pairs}
    mA = np.array(sorted(child_of.keys()))
    matchedB = set(j for _, j in pairs)
    div = []
    for j in range(len(B)):
        if j in matchedB:
            continue
        cand = mA[D[mA, j] <= DIV_MAX_UM]          # yakin, zaten-cocuklu ebeveynler
        best_i, best_d = -1, DIV_MAX_UM + 1
        for i in cand:
            sib = float(np.linalg.norm((B[child_of[i]] - B[j]) * S))   # iki kiz arasi
            if D[i, j] < best_d and sib <= DIV_SIB_UM:
                best_d, best_i = D[i, j], int(i)
        if best_i >= 0:
            div.append((best_i, j))
    return pairs, div


def build_graph(cents, div_on):
    nodes, edges, off, nid = [], [], [], 1
    for t, c in enumerate(cents):
        off.append(nid)
        for p in c:
            nodes.append((nid, t, int(round(p[0])), int(round(p[1])), int(round(p[2])))); nid += 1
    for t in range(len(cents) - 1):
        pairs, div = link_and_divide(cents[t], cents[t + 1], div_on)
        for i, j in pairs:
            edges.append((off[t] + i, off[t + 1] + j))
        for i, j in div:
            edges.append((off[t] + i, off[t + 1] + j))    # bolunme = ebeveynin 2. cikis kenari
    return nodes, edges


def gap_fill(nodes, edges, gap_um=GAP_MAX_UM):
    # 1-kare bosluk kapat: t'de biten track (cocuksuz) + t+2'de baslayan track (ebeveynsiz),
    # <= gap_um ise aradaki t+1'e sentetik node koy, iki kenari kur. GT gerekmez (inference).
    pos = {n[0]: np.array([n[2], n[3], n[4]], float) for n in nodes}
    has_child = set(u for u, v in edges); has_parent = set(v for u, v in edges)
    ends, starts = {}, {}
    for nid, t, z, y, x in nodes:
        if nid not in has_child:
            ends.setdefault(t, []).append(nid)
        if nid not in has_parent:
            starts.setdefault(t, []).append(nid)
    nn, ne = list(nodes), list(edges)
    nxt = max(n[0] for n in nodes) + 1 if nodes else 1
    for f in sorted(ends):
        e, s = ends.get(f, []), starts.get(f + 2, [])
        if not e or not s:
            continue
        A = np.array([pos[i] for i in e]); B = np.array([pos[j] for j in s])
        D = cdist(A * S, B * S)
        cost = np.where(D <= gap_um, D, 1e9)
        r, c = linear_sum_assignment(cost)
        for i, j in zip(r, c):
            if D[i, j] <= gap_um:
                mid = (pos[e[i]] + pos[s[j]]) / 2
                sid = nxt; nxt += 1
                nn.append((sid, f + 1, int(round(mid[0])), int(round(mid[1])), int(round(mid[2]))))
                ne.append((e[i], sid)); ne.append((sid, s[j]))
    return nn, ne


def track_dataset(arr, div_on=None, cents=None, gapfill=None):
    if div_on is None:
        div_on = DIV_ON
    if gapfill is None:
        gapfill = GAPFILL
    if cents is None:
        cents = detect_all(arr)
    nodes, edges = build_graph(cents, div_on)
    if gapfill:
        nodes, edges = gap_fill(nodes, edges)
    return nodes, edges

print('ok')

## 5 — Yerel metrik: **resmi** edge Jaccard + division (metrics.md ile doğrulandı)

Önceki metriğimiz iki yerde resmiden sapıyordu (yerel skoru şişiriyordu):

- **FP kuralı:** Bir tahmin kenarı TP değilse ve **hedefi başka-kaynaklı** ya da **kaynağı başka-hedefli** bir GT node'a eşleşiyorsa **FP** (tek ucu eşleşse bile). Diğerleri yok sayılır. Eskiden tek uç eşleşmeyince atlıyorduk → tek-uçlu FP'leri kaçırıyorduk.
- **Agregasyon:** Ceza **video-başına** uygulanır, sonra `w_i = TP+FP+FN` ile ağırlıklı ortalama. Eskiden global havuzlayıp tek ceza uyguluyorduk.

`T_true` sunucuda (bize verilmiyor — 6. bölümde doğrulanıyor); yerel proxy = referans blob sayımı, penalty ≈ 1. Formül: `adj = raw × max(0, 1−0.1·(T_pred−T_true)/T_true)`, `FINAL = adj + 0.1·div_J`. (Division'ın resmi lineage-bazlı hâli → öneri 6, sonra.)

In [ ]:
def eval_vs_gt(nodes, edges, gt_ndf, gt_edges, t_true=None):
    # RESMI metrik (metrics.md dogrulandi). Dataset basina ham sayaclar doner.
    pn = pd.DataFrame(nodes, columns=['node_id', 't', 'z', 'y', 'x'])
    gmap = {}                                      # pred_id -> gt_id (7 um eslesme)
    for t, g in gt_ndf.groupby('t'):
        p = pn[pn.t == int(t)]
        if len(p) == 0 or len(g) == 0:
            continue
        D = cdist(p[['z', 'y', 'x']].values * S, g[['z', 'y', 'x']].values * S)
        cost = np.where(D <= MATCH_UM, D, 1e6)
        r, c = linear_sum_assignment(cost)
        pid = p['node_id'].values; gid = g['id'].values
        for i, j in zip(r, c):
            if D[i, j] <= MATCH_UM:
                gmap[int(pid[i])] = int(gid[j])
    # GT kenar yapisi: cikan hedefler / gelen kaynaklar (resmi FP kurali icin)
    gtset = set(); gt_out = {}; gt_in = {}
    for u, v in gt_edges:
        u, v = int(u), int(v)
        gtset.add((u, v))
        gt_out.setdefault(u, set()).add(v)
        gt_in.setdefault(v, set()).add(u)
    # RESMI edge siniflama: TP = iki uc da eslesip aralarinda GT kenari var.
    # TP degilse FP: hedef bir baska-kaynakli GT node'a, VEYA kaynak bir baska-hedefli
    # GT node'a eslesiyorsa (tek uc eslesse bile). Digerleri YOK SAYILIR.
    eTP = eFP = 0; cov = set()
    for pu, pv in edges:
        gu, gv = gmap.get(pu), gmap.get(pv)
        if gu is not None and gv is not None and (gu, gv) in gtset:
            eTP += 1; cov.add((gu, gv))
        elif (gu is not None and len(gt_out.get(gu, ())) > 0) or \
             (gv is not None and len(gt_in.get(gv, ())) > 0):
            eFP += 1
        # else: isaretlenmemis bolge -> yok say
    eFN = len(gtset) - len(cov)
    # Division (yaklasik; resmi lineage-bazli tanima yaklastirma -> oneri 6, sonra).
    gt_div = set(u for u, c in Counter(int(u) for u, _ in gt_edges).items() if c >= 2)
    pr_div = set(u for u, c in Counter(u for u, _ in edges).items() if c >= 2)
    gt2pr = {}
    for pid_, gid_ in gmap.items():
        gt2pr.setdefault(gid_, pid_)
    dTP = sum(1 for gd in gt_div if gt2pr.get(gd) in pr_div)
    dFP = sum(1 for pdv in pr_div if (pdv in gmap) and (gmap[pdv] not in gt_div))
    dFN = len(gt_div) - dTP
    return dict(eTP=eTP, eFP=eFP, eFN=eFN, dTP=dTP, dFP=dFP, dFN=dFN,
                jaccard=round(eTP / max(eTP + eFP + eFN, 1), 4),
                node_recall=round(len(set(gmap.values())) / max(len(gt_ndf), 1), 4),
                pred_nodes=len(nodes), T_true=(t_true if t_true is not None else len(nodes)),
                gt_div=len(gt_div))


def micro_final(rows):
    # RESMI agregasyon: ceza VIDEO-BASINA uygulanir, sonra w_i=TP+FP+FN ile agirlikli ort.
    # Division sayaclari havuzlanip Jaccard. FINAL = adj_edge_J + 0.1*div_J.
    num = den = 0.0
    for r in rows:
        w = r['eTP'] + r['eFP'] + r['eFN']
        if w == 0:
            continue
        raw_i = r['eTP'] / w
        tt = max(r['T_true'], 1)
        pen_i = max(0.0, 1.0 - 0.1 * (r['pred_nodes'] - tt) / tt)
        num += w * raw_i * pen_i
        den += w
    adj = num / max(den, 1)
    eTP = sum(r['eTP'] for r in rows); eFP = sum(r['eFP'] for r in rows); eFN = sum(r['eFN'] for r in rows)
    dTP = sum(r['dTP'] for r in rows); dFP = sum(r['dFP'] for r in rows); dFN = sum(r['dFN'] for r in rows)
    raw_pooled = eTP / max(eTP + eFP + eFN, 1)
    dJ = dTP / max(dTP + dFP + dFN, 1)
    return dict(raw_edge_J=round(raw_pooled, 4), adj_edge_J=round(adj, 4),
                div_J=round(dJ, 4), FINAL=round(adj + 0.1 * dJ, 4),
                eTP=eTP, eFP=eFP, eFN=eFN, dTP=dTP, dFP=dFP, dFN=dFN)

print('ok')

### 5a — Sanity: GT → GT skoru **1.0** olmalı (yalnızca DEV)

In [ ]:
if DEV:
    g_ndf, g_edges = load_geff(TRAIN / (test_names[0] + '.geff'))
    gt_nodes = [(int(r.id), int(r.t), float(r.z), float(r.y), float(r.x)) for r in g_ndf.itertuples()]
    chk = eval_vs_gt(gt_nodes, [(int(u), int(v)) for u, v in g_edges], g_ndf, g_edges)
    print('GT->GT:', {k: chk[k] for k in ('jaccard', 'eTP', 'eFP', 'eFN', 'gt_div')})
    assert chk['jaccard'] > 0.999, 'SANITY FAIL — metrik implementasyonu hatali!'
    print('>> Metrik dogrulandi.')
else:
    print('GERCEK RERUN -> sanity atlandi (GT yok)')

## 6 — Yerel skor: submit edilecek konfigürasyon (yalnızca DEV)

Submit edilecek `DIV_ON`/`GAPFILL` konfigürasyonunun 4 placeholder üzerindeki resmi
`adj_edge_J`'si. Detection bir kez. Bu sayı LB'yi mutlak tahmin etmez (placeholder'lar
pesimist) ama **değişikliklerin yönünü** güvenilir gösterir.

In [ ]:
if DEV:
    # Submit edilecek KONFIG'in (DIV_ON, GAPFILL) yerel skoru. Detection bir kez, cache.
    rows = []
    for nm in test_names:
        gp = TRAIN / (nm + '.geff')
        if not gp.exists():
            continue
        t0 = time.time()
        arr = open_image(TEST / (nm + '.zarr'))
        cents = detect_all(arr)
        nodes, edges = build_graph(cents, DIV_ON)
        if GAPFILL:
            nodes, edges = gap_fill(nodes, edges)
        gn, ge = load_geff(gp)
        r = eval_vs_gt(nodes, edges, gn, ge, t_true=estimate_true_count(arr))
        r['dataset'] = nm; rows.append(r)
        print(f'  {nm}: {time.time()-t0:.0f}s')

    m = micro_final(rows)
    print(f'\nKONFIG: DIV_ON={DIV_ON} | GAPFILL={GAPFILL}')
    for r in rows:
        print(f"  {r['dataset']}: rawJ={r['jaccard']:.3f} recall={r['node_recall']:.3f} "
              f"eTP={r['eTP']} eFP={r['eFP']} eFN={r['eFN']} | pred/kare={r['pred_nodes']//100}")
    print(f"  >> adj_edge_J={m['adj_edge_J']} | div_J={m['div_J']} | FINAL={m['FINAL']}  "
          f"[eTP={m['eTP']} eFP={m['eFP']} eFN={m['eFN']}]")
    print('  (referans: baseline 0.6918 -> gap-fill 0.7014 | LB anchor ~0.749)')
else:
    print('GERCEK RERUN -> eval atlandi (sadece submission uretilir)')

## 7 — Gönderim: `test/` dinamik gez, `submission.csv` yaz

Satır satır yaz (gizli test büyük olabilir). Bir dataset patlarsa yer tutucu node koy,
koşuyu öldürme — her test dataset'i submission'da yer almalı.

In [ ]:
import csv
names = test_names if MAX_TEST is None else test_names[:MAX_TEST]
gid = tot_n = tot_e = 0; failed = []; t_start = time.time()
with open(OUT_CSV, 'w', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id'])
    for k, nm in enumerate(names, 1):
        t0 = time.time()
        try:
            nodes, edges = track_dataset(open_image(TEST / (nm + '.zarr')))
        except Exception as e:
            print(f'  [HATA] {nm}: {type(e).__name__}: {e}'); nodes, edges = [], []; failed.append(nm)
        for nid, t, z, y, x in nodes:
            w.writerow([gid, nm, 'node', nid, t, z, y, x, -1, -1]); gid += 1
        for u, v in edges:
            w.writerow([gid, nm, 'edge', -1, -1, -1, -1, -1, u, v]); gid += 1
        if not nodes:
            w.writerow([gid, nm, 'node', 1, 0, 0, 0, 0, -1, -1]); gid += 1
            print(f'  [uyari] {nm}: tespit yok -> yer tutucu')
        tot_n += len(nodes); tot_e += len(edges)
        print(f'[{k}/{len(names)}] {nm}: node={len(nodes)} edge={len(edges)} ({time.time()-t0:.0f}s)')
print(f'\nYAZILDI {OUT_CSV} | satir={gid:,} node={tot_n:,} edge={tot_e:,} | {(time.time()-t_start)/60:.1f} dk')
if failed:
    print('BASARISIZ (yer tutucu kondu):', failed)

## 8 — Şema doğrulama

In [ ]:
head = pd.read_csv(OUT_CSV, nrows=5)
ss = ROOT / 'sample_submission.csv'
if ss.exists():
    assert list(head.columns) == list(pd.read_csv(ss).columns), 'KOLON UYUSMAZLIGI!'
    print('kolonlar OK')
print(head.to_string(index=False))

if DEV:
    sub = pd.read_csv(OUT_CSV)
    missing = set(names) - set(sub.dataset.unique())
    assert not missing, f'eksik dataset: {missing}'
    bad = 0
    for ds, g in sub.groupby('dataset'):
        nid = set(g[g.row_type == 'node'].node_id); e = g[g.row_type == 'edge']
        bad += int((~e.source_id.isin(nid)).sum() + (~e.target_id.isin(nid)).sum())
    assert bad == 0, f'gecersiz edge referansi: {bad}'
    print(f'row_type={dict(sub.row_type.value_counts())} | dataset={sub.dataset.nunique()} | edge-ref OK')
    print('>> Submission gecerli.')
else:
    print('GERCEK RERUN -> agir kontrol atlandi | satir:', sum(1 for _ in open(OUT_CSV)) - 1)
print('\nHatirlatma: Internet = OFF olmali, yoksa submit reddedilir.')

## 9 — Sonraki adımlar

Bu baseline **submit-hazır** (gap-fill dahil, yerel 0.7014). Klasik hattın kolay kaldıraçları
ölçülerek tüketildi. Bundan sonrası:

- [ ] **U-Net detection (YOL B)** — asıl kaldıraç: kaybın çoğu (255 zor-detection FN) klasik
  detection'ın erişemediği yerde. GPU T4 x2 açık. Önce GPU'nun **scored rerun'da da** açık
  olduğunu doğrula (editörde açık ≠ rerun'da açık).
- [ ] **Prefix-bazlı CV** (`train/` 199 dataset, 44b6/6bba stratifiye) → `05_evaluation.ipynb`;
  4 placeholder'a overfit'i önler.
- [ ] **Bölünme (öneri 6)** — lineage-bazlı resmi div metriği + geniş örnekle `DIV_ON` yeniden dene.
- [ ] **Gap-fill tuning** — `GAP_MAX_UM` ve 2-kare boşluğa genişletme, CV'de ölç.